In [ ]:
import os
import glob
import gzip
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import mplhep as hep
from coffea.util import load
import correctionlib.schemav2 as schema

hep.style.use(hep.style.CMS)

# ==========================================
# 1. CONFIGURATION
# ==========================================
YEAR = "2024"  # Options: "2024" (Run 3), "2016_PreVFP", "2016_PostVFP", "2016", "2017", "2018"
COFFEA_DIR = "outputs/nominal/trigeff/"   # first pass: muon trigger SF = MUO reference ("pog")
CATEGORY = "trigeff"

TRIG_PREFIX = "HLT_"

LUMI_DICT = {
    "2016_PreVFP": 19.5, "2016_PostVFP": 16.8, "2017": 41.5, "2018": 59.8, "2016": 19.5 + 16.8,
    "2024": 109.95,
}

TRIGGERS = {
    "2024": {
        "ele": ['Ele30_WPTight_Gsf', 'Ele115_CaloIdVT_GsfTrkIdT'],
        "mu":  ['IsoMu24', 'Mu50', 'HighPtTkMu100', 'CascadeMu100']
    },
    "2016_PreVFP": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24']
    },
    "2016_PostVFP": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24']
    },
    "2016": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24']
    },
    "2017": {
        # NOTE: Ele32_WPTight_Gsf_L1DoubleEG normally needs the TrigObj filter-bit emulation
        # (EGamma recommendation) — check before using 2017 results.
        "ele": ['Ele35_WPTight_Gsf', 'Ele32_WPTight_Gsf_L1DoubleEG', 'Photon200', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu27']
    },
    "2018": {
        "ele": ['Ele32_WPTight_Gsf', 'Photon200', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24']
    }
}

# Only data datasets whose name contains this string are used. The reference trigger is the
# muon trigger, so data must come from the muon primary dataset only; mixing in SingleElectron /
# EGamma double-counts events that fire both triggers. Set to None to disable the filter.
REF_DATA_PD = "Muon"

# Bootstrap settings for alpha and the fake systematic
N_BOOT = 200
# If True, the fake systematic is sqrt(max(0, delta^2 - sigma_stat(delta)^2)) so MC stat noise
# in the nominal-vs-alternate difference isn't counted as a systematic (it's already in the stat unc.).
SUBTRACT_STAT_FROM_FAKE_SYST = True

# --- DERIVED ---
LUMI = LUMI_DICT[YEAR]
ele_triggers = [f"{TRIG_PREFIX}{t}" for t in TRIGGERS[YEAR]["ele"]]
mu_triggers = [f"{TRIG_PREFIX}{t}" for t in TRIGGERS[YEAR]["mu"]]
trigger_vars = mu_triggers + ele_triggers

all_processes = ['TTBar', 'data_obs', 'tt_B', 'TTBar_SemiLep', 'tt_B_SemiLep']

# MC reweighting: muon trigger SF is applied because the muon trigger is the reference
weight_vars = [
    'events_genWeight', 'events_norm_weight', 'events_topptWeight',
    'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf', 'events_mu_iso_sf',
    'events_puWeight', 'events_mu_trig_sf', 'events_btag_sf'
]
if not YEAR.startswith("202"):   # L1 prefiring weight: Run 2 only
    weight_vars.append('events_l1prefire_sf')

validation_vars = ['events_ele_pt', 'events_ele_eta', 'events_n_ak4jets', 'events_n_b_outZH'] + trigger_vars
vars_to_extract = validation_vars + weight_vars

# --- BINNING ---
USE_ABS_ETA = True
PT_BINS = np.array([30, 40, 50, 70, 120, 500])
ETA_BINS_SIGNED = np.array([-2.5, -1.5, -0.8, 0.0, 0.8, 1.5, 2.5])
ETA_BINS = np.array([0.0, 0.8, 1.5, 2.5]) if USE_ABS_ETA else ETA_BINS_SIGNED
ETA_COL = 'ele_abseta' if USE_ABS_ETA else 'events_ele_eta'
ETA_LABEL = r"Electron $|\eta|$" if USE_ABS_ETA else r"Electron $\eta$"
ETA_BINS_SYMCHECK = ETA_BINS if USE_ABS_ETA else ETA_BINS_SIGNED[ETA_BINS_SIGNED >= 0]
N_PT, N_ETA = len(PT_BINS) - 1, len(ETA_BINS) - 1

# Minimum effective denominator entries (in BOTH data and MC) for a bin to be trusted
MIN_EFF_STATS = 5

# 68.27% central interval
CL_LOW, CL_HIGH = 0.3173 / 2, 1 - 0.3173 / 2


# ==========================================
# 2. DATA EXTRACTION
# ==========================================
def getZhbbWeight(df_):
    if 'events_norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index)

    weight = df_['events_norm_weight'].copy()
    gen_w = df_.get('events_genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)

    sfs = [
        'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf',
        'events_mu_iso_sf', 'events_puWeight', 'events_topptWeight', 'events_mu_trig_sf', 'events_btag_sf',
    ]
    if not YEAR.startswith("202"):   # UL NanoAOD L1 prefiring weight (Run 2 only)
        sfs.append('events_l1prefire_sf')

    for sf in sfs:
        if sf not in df_.columns:
            print(f"  WARNING: weight column '{sf}' missing, using 1.0")
        weight *= df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)

    return weight


def map_process(raw_proc):
    if 'data' in raw_proc.lower():
        return 'data_obs'
    if 'TTTo2L2Nu' in raw_proc:
        return 'TTBar' if 'tt+B' not in raw_proc else None
    if 'TTbb_2L2Nu' in raw_proc:
        return 'tt_B' if 'tt+B' in raw_proc else None
    if 'TTToSemiLeptonic' in raw_proc:
        return 'TTBar_SemiLep' if 'tt+B' not in raw_proc else None
    if 'TTbb_SemiLeptonic' in raw_proc:
        return 'tt_B_SemiLep' if 'tt+B' in raw_proc else None
    return None


def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR):
    tracked_data = {proc: {} for proc in all_processes}
    files_per_dataset = {}
    skipped_data = set()
    missing_genweight = set()

    # run_pipeline.py layout: <outdir>/nominal/trigeff/output_all.coffea
    all_files = glob.glob(os.path.join(coffea_dir, "output_all.coffea"))
    valid_files = all_files
    print(f"Extracting '{variation}' from {len(valid_files)} matching files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})

        for raw_proc in filein['columns'].keys():
            mapped_proc = map_process(raw_proc)
            if mapped_proc not in all_processes:
                continue
            is_data = mapped_proc == 'data_obs'

            for dataset in filein['columns'][raw_proc].keys():
                if YEAR not in dataset:
                    continue
                if is_data and REF_DATA_PD and REF_DATA_PD not in dataset and REF_DATA_PD not in raw_proc:
                    skipped_data.add(dataset)
                    continue

                genweight = 1.0
                if not is_data:
                    if dataset not in genweight_dict:
                        missing_genweight.add(dataset)
                    genweight = genweight_dict.get(dataset, 1.0)
                    if isinstance(genweight, dict):
                        genweight = genweight.get(dataset, 1.0)

                try:
                    base_dict = filein['columns'][raw_proc][dataset][CATEGORY][variation]
                except KeyError:
                    continue

                tmp_data = {}
                for var in (validation_vars if is_data else vars_to_extract):
                    if var not in base_dict:
                        continue
                    arr = np.array(base_dict[var].value)
                    tmp_data[var] = arr / genweight if (var == 'events_norm_weight' and not is_data) else arr

                if tmp_data:
                    df = pd.DataFrame(tmp_data)
                    df['dataset'] = dataset
                    tracked_data[mapped_proc].setdefault(dataset, []).append(df)
                    files_per_dataset.setdefault((mapped_proc, dataset), set()).add(os.path.basename(file_path))

    # --- Loader sanity report ---
    print("\nDatasets loaded:")
    for (proc, dataset), files in sorted(files_per_dataset.items()):
        print(f"  {proc:14s} {dataset}  ({len(files)} file(s))")
        if proc != 'data_obs' and len(files) > 1:
            print("    WARNING: MC dataset spread over several files. norm_weight is divided by each file's "
                  "'sum_signOf_genweights'; make sure that is the FULL-dataset sum, not a per-file partial sum.")
    if skipped_data:
        print(f"\nSkipped data datasets (not matching REF_DATA_PD='{REF_DATA_PD}'): {sorted(skipped_data)}")
    if missing_genweight:
        print(f"\nWARNING: no sum_signOf_genweights entry for {sorted(missing_genweight)} -> genweight=1 used!")
    if not tracked_data['data_obs']:
        raise RuntimeError(f"No data datasets matched REF_DATA_PD='{REF_DATA_PD}'. "
                           f"Skipped: {sorted(skipped_data)}. Adjust REF_DATA_PD or set it to None.")

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = [df for branches in tracked_data[proc].values() for df in branches]
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df['dataset'] = df['dataset'].astype('category')
            df['tot_weight'] = getZhbbWeight(df) if proc != 'data_obs' else 1.0
            df['ele_abseta'] = np.abs(df['events_ele_eta'])
            data_dict[proc] = df
        else:
            data_dict[proc] = None
    return data_dict


def build_mc_samples(data_dict):
    dilep = [data_dict[p] for p in ['TTBar', 'tt_B'] if data_dict.get(p) is not None]
    semi = [data_dict[p] for p in ['TTBar_SemiLep', 'tt_B_SemiLep'] if data_dict.get(p) is not None]
    if not dilep:
        raise RuntimeError("No dileptonic ttbar MC loaded.")
    data_dict['MC_Combined'] = pd.concat(dilep, ignore_index=True)
    parts = [d.assign(is_semi=False) for d in dilep] + [d.assign(is_semi=True) for d in semi]
    data_dict['MC_WithFakes'] = pd.concat(parts, ignore_index=True)
    if not semi:
        print("WARNING: no semileptonic MC loaded; fake systematic will be zero.")


# ==========================================
# 3. HELPERS
# ==========================================
def trigger_masks(df):
    """NaN-safe ORs. A trigger branch missing for some events counts as 'did not fire'."""
    ref = df.reindex(columns=mu_triggers).fillna(False).astype(bool).any(axis=1).values
    tgt = df.reindex(columns=ele_triggers).fillna(False).astype(bool).any(axis=1).values
    acc = (df['events_ele_pt'].between(PT_BINS[0], PT_BINS[-1]) &
           df[ETA_COL].between(ETA_BINS[0], ETA_BINS[-1])).values
    return ref, tgt, acc


def bin_index(df):
    """Flat (pT, eta) bin index per event, -1 if outside the grid. Upper edge goes into the last bin."""
    pt, eta = df['events_ele_pt'].values, df[ETA_COL].values
    ipt = np.searchsorted(PT_BINS, pt, side='right') - 1
    ieta = np.searchsorted(ETA_BINS, eta, side='right') - 1
    ipt[pt == PT_BINS[-1]] = N_PT - 1
    ieta[eta == ETA_BINS[-1]] = N_ETA - 1
    ok = (ipt >= 0) & (ipt < N_PT) & (ieta >= 0) & (ieta < N_ETA)
    return np.where(ok, ipt * N_ETA + ieta, -1)


def hist2d(idx, w, mask):
    m = mask & (idx >= 0)
    return np.bincount(idx[m], weights=w[m], minlength=N_PT * N_ETA).reshape(N_PT, N_ETA)


def safe_ratio(a, b):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(b > 0, a / b, np.nan)


def efficiency_with_cp(h_num, h_den, h_den_w2):
    """Efficiency with Clopper-Pearson interval evaluated at the effective number of entries."""
    with np.errstate(divide='ignore', invalid='ignore'):
        eff = np.where(h_den > 0, h_num / h_den, 0.0)
        n_eff = np.where(h_den_w2 > 0, h_den**2 / h_den_w2, 0.0)
        k_eff = np.clip(eff, 0.0, 1.0) * n_eff
        lo = stats.beta.ppf(CL_LOW, k_eff, n_eff - k_eff + 1)
        hi = stats.beta.ppf(CL_HIGH, k_eff + 1, n_eff - k_eff)
    valid = h_den > 0
    err_up = np.where(valid, np.nan_to_num(hi - eff, nan=0.0), 0.0)
    err_dn = np.where(valid, np.nan_to_num(eff - lo, nan=0.0), 0.0)
    return eff, err_up, err_dn, n_eff


def alpha_from_masks(w, ref, tgt, base):
    S = lambda m: w[m & base].sum()
    return S(ref) * S(tgt) / (S(ref & tgt) * S(base))


# ==========================================
# 4. DIAGNOSTICS
# ==========================================
def trigger_report(data_dict, procs=('data_obs', 'MC_Combined')):
    print("\n=== Trigger branch availability (% of events with branch MISSING, per dataset) ===")
    for proc in procs:
        df = data_dict.get(proc)
        if df is None:
            continue
        miss = pd.DataFrame({t: (df[t].isna() if t in df.columns else pd.Series(True, index=df.index))
                             for t in trigger_vars})
        tab = 100 * miss.groupby(df['dataset'], observed=True).mean()
        if (tab.values > 0).any():
            print(f"[{proc}]")
            print(tab.round(2).to_string())
        else:
            print(f"[{proc}] all trigger branches present in every dataset.")

    print("\n=== Per-path firing rate [%] among reference-triggered events in acceptance ===")
    rows = {}
    for proc in procs:
        df = data_dict.get(proc)
        if df is None:
            continue
        ref, _, acc = trigger_masks(df)
        sel = ref & acc
        w = np.ones(len(df)) if proc == 'data_obs' else df['tot_weight'].values
        rows[proc] = {t: (np.average(df[t].fillna(False).astype(bool).values[sel], weights=w[sel])
                          if t in df.columns else np.nan) for t in ele_triggers}
    print((100 * pd.DataFrame(rows)).round(2).to_string())


def check_eta_symmetry(df):
    """Compare data efficiency for eta>0 vs eta<0 in each |eta| bin."""
    ref, tgt, _ = trigger_masks(df)
    both = ref & tgt
    pt, eta = df['events_ele_pt'].values, df['events_ele_eta'].values
    eff, err = {}, {}
    for side, sel in [('pos', eta > 0), ('neg', eta < 0)]:
        h_d, _, _ = np.histogram2d(pt[ref & sel], np.abs(eta[ref & sel]), bins=[PT_BINS, ETA_BINS_SYMCHECK])
        h_n, _, _ = np.histogram2d(pt[both & sel], np.abs(eta[both & sel]), bins=[PT_BINS, ETA_BINS_SYMCHECK])
        with np.errstate(divide='ignore', invalid='ignore'):
            e = np.where(h_d > 0, h_n / h_d, np.nan)
            eff[side] = e
            err[side] = np.sqrt(np.maximum(e * (1 - e), 1.0 / np.maximum(h_d, 1)) / np.maximum(h_d, 1))
    with np.errstate(divide='ignore', invalid='ignore'):
        pull = (eff['pos'] - eff['neg']) / np.sqrt(err['pos']**2 + err['neg']**2)
    print("\nEta symmetry pull (eff[+eta] - eff[-eta]) / sigma  (rows = pT bins, cols = |eta| bins):")
    print(np.round(pull, 2))
    return pull, eff, err


def eta_symmetry_tests(pull):
    ok = np.isfinite(pull)
    chi2, ndf = np.sum(pull[ok]**2), ok.sum()
    p_global = stats.chi2.sf(chi2, ndf)
    print(f"\n[1] Global chi2/ndf = {chi2:.1f}/{ndf}, p = {p_global:.3f}  -> {'OK' if p_global > 0.05 else 'CHECK'}")
    for j in range(pull.shape[1]):
        col = pull[:, j][np.isfinite(pull[:, j])]
        if len(col) == 0:
            continue
        limit = 2 / np.sqrt(len(col))
        print(f"[2] |eta| [{ETA_BINS_SYMCHECK[j]}, {ETA_BINS_SYMCHECK[j+1]}]: mean pull = {col.mean():+.2f} "
              f"(limit ±{limit:.2f}) -> {'OK' if abs(col.mean()) < limit else 'CHECK'}")
    n_extreme = np.sum(np.abs(pull[ok]) > 3)
    print(f"[3] Bins with |pull| > 3: {n_extreme}  -> {'OK' if n_extreme == 0 else 'CHECK'}")


def alpha_vs_pt(df, alpha_incl, n_boot=100, seed=11):
    """Check that alpha is flat in pT, justifying a single inclusive value."""
    ref, tgt, acc = trigger_masks(df)
    w, pt = df['tot_weight'].values, df['events_ele_pt'].values
    rng = np.random.default_rng(seed)
    print(f"\nAlpha vs electron pT (inclusive alpha = {alpha_incl:.4f}):")
    for lo, hi in zip(PT_BINS[:-1], PT_BINS[1:]):
        m = acc & (pt >= lo) & (pt < hi)
        if not m.any():
            continue
        wm, rm, tm, base = w[m], ref[m], tgt[m], np.ones(m.sum(), bool)
        a = alpha_from_masks(wm, rm, tm, base)
        s = np.std([alpha_from_masks(wm * rng.poisson(1.0, len(wm)), rm, tm, base) for _ in range(n_boot)])
        print(f"  pT [{lo:.0f}, {hi:.0f}): alpha = {a:.4f} ± {s:.4f}   (rough pull vs inclusive: {(a - alpha_incl) / s:+.1f})")


def print_uncertainty_breakdown(sf, comps, valid):
    print("\n=== Uncertainty breakdown [% of SF] (rows = pT bins, cols = eta bins) ===")
    for name, m in comps.items():
        with np.errstate(divide='ignore', invalid='ignore'):
            rel = np.where(valid & (sf > 0), 100 * m / sf, np.nan)
        print(f"\n{name}:")
        print(np.round(rel, 2))


# ==========================================
# 5. MEASUREMENT
# ==========================================
def calculate_trigger_sf(data_dict, target_mc='MC_Combined', target_data='data_obs', compute_alpha=True):
    print(f"\n--- Calculating Trigger Efficiencies & Systematics for {target_mc} ---")
    results = {}
    for process in [target_data, target_mc]:
        df = data_dict.get(process)
        if df is None or df.empty:
            raise RuntimeError(f"No events for {process}")

        ref, tgt, acc = trigger_masks(df)
        idx = bin_index(df)
        w = np.ones(len(df)) if process == target_data else df['tot_weight'].values

        h_den = hist2d(idx, w, ref)
        h_den_w2 = hist2d(idx, w**2, ref)
        h_num = hist2d(idx, w, ref & tgt)
        eff, err_up, err_dn, n_eff = efficiency_with_cp(h_num, h_den, h_den_w2)

        results[process] = {'eff': eff, 'stat_err_up': err_up, 'stat_err_dn': err_dn, 'n_eff': n_eff}
        print(f"Calculated efficiency for {process}. Mean eff over filled bins: {eff[h_den > 0].mean():.3f}")

        if process == target_mc and compute_alpha:
            results['alpha'] = alpha_from_masks(w, ref, tgt, acc)
            rng = np.random.default_rng(42)
            boots = [alpha_from_masks(w * rng.poisson(1.0, len(w)), ref, tgt, acc) for _ in range(N_BOOT)]
            results['alpha_err'] = np.std(boots)
            print(f"Inclusive alpha = {results['alpha']:.4f} ± {results['alpha_err']:.4f} (stat)")

    alpha = results.get('alpha', 1.0)

    with np.errstate(divide='ignore', invalid='ignore'):
        eff_data, eff_mc = results[target_data]['eff'], results[target_mc]['eff']
        valid = (results[target_mc]['n_eff'] >= MIN_EFF_STATS) & (results[target_data]['n_eff'] >= MIN_EFF_STATS) & (eff_mc > 0)

        sf_map = np.where(valid, eff_data / eff_mc, 0.0)

        rel_up_data = np.where(eff_data > 0, results[target_data]['stat_err_up'] / eff_data, 0)
        rel_dn_data = np.where(eff_data > 0, results[target_data]['stat_err_dn'] / eff_data, 0)
        rel_up_mc = np.where(eff_mc > 0, results[target_mc]['stat_err_up'] / eff_mc, 0)
        rel_dn_mc = np.where(eff_mc > 0, results[target_mc]['stat_err_dn'] / eff_mc, 0)

        # SF up: data up / MC down ; SF down: data down / MC up
        sf_stat_up = np.where(valid, sf_map * np.sqrt(rel_up_data**2 + rel_dn_mc**2), 0.0)
        sf_stat_dn = np.where(valid, sf_map * np.sqrt(rel_dn_data**2 + rel_up_mc**2), 0.0)

        alpha_unc = np.where(valid, np.abs(1.0 - alpha) * sf_map, 0.0)

    low_stat = ~valid
    if low_stat.any():
        for i, j in zip(*np.where(low_stat)):
            print(f"  -> LOW STATS: pT [{PT_BINS[i]}, {PT_BINS[i+1]}], eta [{ETA_BINS[j]}, {ETA_BINS[j+1]}]: "
                  f"n_eff data = {results[target_data]['n_eff'][i, j]:.1f}, "
                  f"MC = {results[target_mc]['n_eff'][i, j]:.1f}. Consider merging.")
        # Conservative placeholder while iterating on binning. The final binning should have no flagged bins.
        sf_map[low_stat] = 1.0
        sf_stat_up[low_stat] = 1.0
        sf_stat_dn[low_stat] = 1.0
        alpha_unc[low_stat] = 0.0

    return results, sf_map, sf_stat_up, sf_stat_dn, alpha_unc, valid

def eta_asym_syst(eff, err, eff_folded, sf):
    d, s2 = eff['pos'] - eff['neg'], err['pos']**2 + err['neg']**2
    w = np.where(np.isfinite(d) & (s2 > 0), 1.0 / s2, 0.0)
    d_col = (w * np.nan_to_num(d)).sum(axis=0) / w.sum(axis=0)     # weighted mean over pT
    d_col_err = 1.0 / np.sqrt(w.sum(axis=0))
    for j, (dm, de) in enumerate(zip(d_col, d_col_err)):
        print(f"|eta| [{ETA_BINS[j]}, {ETA_BINS[j+1]}]: eps(+) - eps(-) = {dm:+.4f} ± {de:.4f}")
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(eff_folded > 0, sf * (np.abs(d_col) / 2)[None, :] / eff_folded, 0.0)


def fake_systematic(data_dict, eff_data, valid, n_boot=N_BOOT, seed=7):
    """SF(dilep MC) - SF(dilep+semilep MC), with the MC-stat noise of that difference
    estimated by a Poisson bootstrap applied coherently to both MC definitions."""
    mc = data_dict['MC_WithFakes']
    if not mc['is_semi'].any():
        return np.zeros_like(eff_data)

    ref, tgt, _ = trigger_masks(mc)
    both, idx = ref & tgt, bin_index(mc)
    dil = ~mc['is_semi'].values
    w = mc['tot_weight'].values

    def delta(wt):
        e_nom = safe_ratio(hist2d(idx, wt, both & dil), hist2d(idx, wt, ref & dil))
        e_alt = safe_ratio(hist2d(idx, wt, both), hist2d(idx, wt, ref))
        return safe_ratio(eff_data, e_nom) - safe_ratio(eff_data, e_alt)

    d = delta(w)
    rng = np.random.default_rng(seed)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        boots = np.array([delta(w * rng.poisson(1.0, len(w))) for _ in range(n_boot)])
        sigma = np.nanstd(boots, axis=0)

    semi_frac = 1 - hist2d(idx, w, ref & dil).sum() / hist2d(idx, w, ref).sum()
    print(f"\nSemileptonic fraction of MC denominator (weighted): {semi_frac:.2%}")
    print("Fake-syst significance, delta / sigma_stat(delta) (rows = pT bins, cols = eta bins):")
    with np.errstate(divide='ignore', invalid='ignore'):
        print(np.round(np.where(valid, d / sigma, np.nan), 1))

    if SUBTRACT_STAT_FROM_FAKE_SYST:
        syst = np.sqrt(np.maximum(d**2 - sigma**2, 0.0))
    else:
        syst = np.abs(d)
    return np.where(valid & np.isfinite(syst), syst, 0.0)


# ==========================================
# 6. WORKFLOW
# ==========================================
data_dict = load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR)
build_mc_samples(data_dict)

trigger_report(data_dict)

pull, eff_side, err_side = check_eta_symmetry(data_dict['data_obs'])
eta_symmetry_tests(pull)

results_nom, sf_nominal, sf_stat_up, sf_stat_dn, sf_alpha_err, valid = calculate_trigger_sf(data_dict, target_mc='MC_Combined')
alpha_vs_pt(data_dict['MC_Combined'], results_nom['alpha'])

sf_fake_err = fake_systematic(data_dict, results_nom['data_obs']['eff'], valid)
sf_eta_err = eta_asym_syst(eff_side, err_side, results_nom['data_obs']['eff'], sf_nominal)
sf_syst = np.sqrt(sf_alpha_err**2 + sf_fake_err**2 + sf_eta_err**2)

sf_total_err_up = np.sqrt(sf_stat_up**2 + sf_syst**2)
sf_total_err_dn = np.sqrt(sf_stat_dn**2 + sf_syst**2)

print_uncertainty_breakdown(sf_nominal, {
    'stat up': sf_stat_up, 'stat down': sf_stat_dn,
    'alpha (correlation)': sf_alpha_err, 'fake (semilep)': sf_fake_err,
    'eta asymmetry': sf_eta_err,
    'total up': sf_total_err_up, 'total down': sf_total_err_dn,
}, valid)

print("\nData n_eff per bin (rows = pT bins, cols = eta bins):")
print(np.round(results_nom['data_obs']['n_eff'], 0))
print("MC n_eff per bin (rows = pT bins, cols = eta bins):")
print(np.round(results_nom['MC_Combined']['n_eff'], 0))


# ==========================================
# 7. PLOTTING & EXPORT
# ==========================================
def plot_annotated_heatmap(val, err_up, err_dn, title, filename, is_sf=False, low_stat=None, data_label=True):
    fig, ax = plt.subplots(figsize=(14, 8))
    X, Y = np.meshgrid(PT_BINS, ETA_BINS)

    shown = val.astype(float).copy()
    if low_stat is not None:
        shown[low_stat] = np.nan

    if is_sf:
        dev = np.nanmax(np.abs(shown - 1.0)) if np.isfinite(shown).any() else 0.1
        half = max(0.05, dev)
        vmin, vmax, cmap, label = 0.0, 1.0, 'viridis', "SF (Data/MC)"
    else:
        vmin, vmax, cmap, label = 0.0, 1.0, 'viridis', "Efficiency"

    im = ax.pcolormesh(X, Y, np.ma.masked_invalid(shown).T, cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(label, fontsize=18)

    ax.set_xscale('log')
    ax.set_xticks(PT_BINS)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([f"{pt:.0f}" for pt in PT_BINS])
    ax.minorticks_off()
    ax.set_xlabel(r"Electron $p_T$ [GeV]", fontsize=18)
    ax.set_yticks(ETA_BINS)
    ax.set_yticklabels([f"{eta:.1f}" for eta in ETA_BINS])
    ax.set_ylabel(ETA_LABEL, fontsize=18)

    for i in range(N_PT):
        for j in range(N_ETA):
            x_c = np.exp((np.log(PT_BINS[i]) + np.log(PT_BINS[i + 1])) / 2.0)
            y_c = (ETA_BINS[j] + ETA_BINS[j + 1]) / 2.0
            if low_stat is not None and low_stat[i, j]:
                ax.text(x_c, y_c, "low\nstats", ha='center', va='center', color='gray', fontsize=10)
                continue
            if val[i, j] <= 0:
                continue
            if is_sf:
                color = 'black'
            else:
                color = 'white' if (val[i, j] - vmin) / (vmax - vmin) < 0.5 else 'black'
            text = f"{val[i, j]:.3f}\n+{err_up[i, j]:.3f}\n-{err_dn[i, j]:.3f}"
            ax.text(x_c, y_c, text, ha='center', va='center', color=color, fontsize=10, fontweight='bold')

    fig.suptitle(title, fontsize=22, fontweight='bold', y=0.94)
    hep.cms.label("Preliminary", data=data_label, lumi=LUMI, ax=ax)
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()


low_stat = ~valid
print("\nGenerating Heatmaps...")
plot_annotated_heatmap(results_nom['data_obs']['eff'], results_nom['data_obs']['stat_err_up'],
                       results_nom['data_obs']['stat_err_dn'], f"Data Electron Trigger Efficiency ({YEAR})",
                       f"pdf/trigger/Data_Ele_Eff_{YEAR}.pdf", low_stat=low_stat)
plot_annotated_heatmap(results_nom['MC_Combined']['eff'], results_nom['MC_Combined']['stat_err_up'],
                       results_nom['MC_Combined']['stat_err_dn'], f"MC Electron Trigger Efficiency ({YEAR})",
                       f"pdf/trigger/MC_Ele_Eff_{YEAR}.pdf", low_stat=low_stat, data_label=False)
plot_annotated_heatmap(sf_nominal, sf_total_err_up, sf_total_err_dn, f"Electron Trigger Scale Factors ({YEAR})",
                       f"pdf/trigger/Ele_SF_{YEAR}.pdf", is_sf=True, low_stat=low_stat)

ok = np.isfinite(pull); chi2, ndf = np.sum(pull[ok]**2), ok.sum()
col_mean, n_col = np.nanmean(pull, axis=0), np.isfinite(pull).sum(axis=0)
rel = lambda m: 100 * np.where(valid, m / sf_nominal, np.nan)
print(f"\n=== AN numbers ({YEAR}) ===")
print(f"chi2/ndf = {chi2:.1f}/{ndf}, p = {stats.chi2.sf(chi2, ndf):.2f}, max|d| = {np.abs(pull[ok]).max():.2f}")
print("mean pull per |eta| region: " + ", ".join(f"{m:+.2f} ({m*np.sqrt(n):+.1f} sigma)" for m, n in zip(col_mean, n_col)))
print(f"alpha = {results_nom['alpha']:.4f} ± {results_nom['alpha_err']:.4f}")
for name, m in [('stat up', sf_stat_up), ('stat down', sf_stat_dn), ('fake', sf_fake_err), ('eta asym', sf_eta_err)]:
    print(f"{name:10s}: {np.nanmin(rel(m)):.2f}% – {np.nanmax(rel(m)):.2f}%")

def export_to_correctionlib():
    input_pt = schema.Variable(name="pt", type="real", description="Electron pT")
    input_eta = schema.Variable(name="eta", type="real", description="Electron pseudorapidity (signed)")
    input_syst = schema.Variable(
        name="systematic", type="string",
        description="'nominal', 'up'/'down' (total), 'stat_up'/'stat_down', 'syst_up'/'syst_down' "
                    "(alpha + semilep fake, symmetric). Stat and syst can be decorrelated across years."
    )

    variations = {
        "nominal": sf_nominal,
        "up": sf_nominal + sf_total_err_up,
        "down": np.clip(sf_nominal - sf_total_err_dn, 0.0, None),
        "stat_up": sf_nominal + sf_stat_up,
        "stat_down": np.clip(sf_nominal - sf_stat_dn, 0.0, None),
        "syst_up": sf_nominal + sf_syst,
        "syst_down": np.clip(sf_nominal - sf_syst, 0.0, None),
    }

    def build_multibin(val_matrix):
        # If measured in |eta|, mirror back to signed eta so consumers pass signed eta
        if USE_ABS_ETA:
            edges_eta = np.concatenate([-ETA_BINS[:0:-1], ETA_BINS])
            content = np.concatenate([val_matrix[:, ::-1], val_matrix], axis=1)
        else:
            edges_eta, content = ETA_BINS, val_matrix
        return schema.MultiBinning(nodetype="multibinning", inputs=["pt", "eta"],
                                   edges=[PT_BINS.tolist(), edges_eta.tolist()],
                                   content=content.flatten().tolist(), flow="clamp")

    cat = schema.Category(nodetype="category", input="systematic", content=[
        schema.CategoryItem(key=k, value=build_multibin(v)) for k, v in variations.items()
    ])

    desc = (f"Electron trigger SF {YEAR}, muon-reference (orthogonal) method in ttbar e-mu. "
            f"Inclusive alpha = {results_nom['alpha']:.4f} ± {results_nom['alpha_err']:.4f}.")
    cset = schema.CorrectionSet(schema_version=2, description=desc, corrections=[
        schema.Correction(name="EleTrigSF", description=desc, version=1,
                          inputs=[input_pt, input_eta, input_syst],
                          output=schema.Variable(name="weight", type="real"), data=cat)
    ])

    fname = f"ele_trig_sf_{YEAR}.json.gz"
    with gzip.open(fname, "wt") as fout:
        fout.write(cset.json(exclude_unset=True))
    print(f"\nSaved correctionlib to: {fname}")


export_to_correctionlib()

In [ ]:
import correctionlib
c = correctionlib.CorrectionSet.from_file(f"ele_trig_sf_{YEAR}.json.gz")["EleTrigSF"]
for pt, eta in [(35, 0.3), (35, -0.3), (45, 2.0), (45, -2.0), (600, 2.4), (100, 2.6)]:
    print(pt, eta, [round(c.evaluate(float(pt), float(eta), s), 3) for s in ("nominal", "up", "down")])

In [ ]:
import os
import glob
import gzip
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import mplhep as hep
from coffea.util import load
import correctionlib.schemav2 as schema

hep.style.use(hep.style.CMS)

# ==========================================
# 1. CONFIGURATION
# ==========================================
YEAR = "2024"  # Options: "2024" (Run 3), "2016_PreVFP", "2016_PostVFP", "2016", "2017", "2018"
COFFEA_DIR = "outputs/nominal/trigeff/"   # second pass: rerun trigeff with the electron SF from the first pass
CATEGORY = "trigeff"

TRIG_PREFIX = "HLT_"

LUMI_DICT = {
    "2016_PreVFP": 19.5, "2016_PostVFP": 16.8, "2016": 19.5 + 16.8, "2017": 41.5, "2018": 59.8,
    "2024": 109.95,
}

TRIGGERS = {
    "2024": {
        "ele": ['Ele30_WPTight_Gsf', 'Ele115_CaloIdVT_GsfTrkIdT'],
        "mu":  ['IsoMu24', 'Mu50', 'HighPtTkMu100', 'CascadeMu100']
    },
    "2016_PreVFP": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24', 'Mu50', 'TkMu50']
    },
    "2016_PostVFP": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24', 'Mu50', 'TkMu50']
    },
    "2016": {
        "ele": ['Ele27_WPTight_Gsf', 'Photon175', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'IsoTkMu24', 'Mu50', 'TkMu50']
    },
    "2017": {
        "ele": ['Ele35_WPTight_Gsf', 'Ele32_WPTight_Gsf_L1DoubleEG', 'Photon200', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu27', 'Mu50', 'OldMu100', 'TkMu100']
    },
    "2018": {
        "ele": ['Ele32_WPTight_Gsf', 'Photon200', 'Ele115_CaloIdVT_GsfTrkIdT', 'Ele50_CaloIdVT_GsfTrkIdT_PFJet165'],
        "mu":  ['IsoMu24', 'Mu50', 'OldMu100', 'TkMu100']
    }
}

# The reference trigger is the electron trigger, so data must come from the electron primary
# dataset only (SingleElectron in 2016/2017, EGamma in 2018). Mixing in SingleMuon or MuonEG
# double-counts events that fire both triggers. Set a year's entry to None to disable the filter.
REF_DATA_PD_DICT = {
    "2016_PreVFP": "SingleElectron", "2016_PostVFP": "SingleElectron", "2016": "EGamma",
    "2017": "EGamma", "2018": "EGamma", "2024": "EGamma",
}

# Data datasets are not filtered by YEAR: COFFEA_DIR is era-specific, and data dataset names
# may not contain the era string (e.g. "2016_PreVFP"). MC datasets are filtered by YEAR.
FILTER_DATA_BY_YEAR = False

# Bootstrap settings for alpha and the fake systematic
N_BOOT = 200
# If True, the fake systematic is sqrt(max(0, delta^2 - sigma_stat(delta)^2)) so MC stat noise
# in the nominal-vs-alternate difference isn't counted as a systematic (it's already in the stat unc.).
SUBTRACT_STAT_FROM_FAKE_SYST = True

# --- DERIVED ---
LUMI = LUMI_DICT[YEAR]
REF_DATA_PD = REF_DATA_PD_DICT[YEAR]
ele_triggers = [f"{TRIG_PREFIX}{t}" for t in TRIGGERS[YEAR]["ele"]]   # reference
mu_triggers = [f"{TRIG_PREFIX}{t}" for t in TRIGGERS[YEAR]["mu"]]     # target
trigger_vars = mu_triggers + ele_triggers

all_processes = ['TTBar', 'data_obs', 'tt_B', 'TTBar_SemiLep', 'tt_B_SemiLep']

# MC reweighting: the electron trigger SF (from the electron measurement) is applied because the
# electron trigger is the reference here.
weight_vars = [
    'events_genWeight', 'events_norm_weight', 'events_topptWeight',
    'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf', 'events_mu_iso_sf',
    'events_puWeight', 'events_ele_trig_sf', 'events_btag_sf'
]
if not YEAR.startswith("202"):   # L1 prefiring weight: Run 2 only
    weight_vars.append('events_l1prefire_sf')

PT_COL, ETA_RAW = 'events_muon_pt', 'events_muon_eta'
validation_vars = [PT_COL, ETA_RAW, 'events_n_ak4jets', 'events_n_b_outZH'] + trigger_vars
vars_to_extract = validation_vars + weight_vars

# --- BINNING ---
USE_ABS_ETA = True
PT_BINS = np.array([29, 40, 50, 60, 120, 200, 500])
ETA_BINS_SIGNED = np.array([-2.4, -1.2, -0.9, 0.0, 0.9, 1.2, 2.4])
ETA_BINS = np.array([0.0, 0.9, 1.2, 2.4]) if USE_ABS_ETA else ETA_BINS_SIGNED
ETA_COL = 'mu_abseta' if USE_ABS_ETA else ETA_RAW
ETA_LABEL = r"Muon $|\eta|$" if USE_ABS_ETA else r"Muon $\eta$"
ETA_BINS_SYMCHECK = ETA_BINS if USE_ABS_ETA else ETA_BINS_SIGNED[ETA_BINS_SIGNED >= 0]
N_PT, N_ETA = len(PT_BINS) - 1, len(ETA_BINS) - 1

# Minimum effective denominator entries (in BOTH data and MC) for a bin to be trusted
MIN_EFF_STATS = 5

# 68.27% central interval
CL_LOW, CL_HIGH = 0.3173 / 2, 1 - 0.3173 / 2


# ==========================================
# 2. DATA EXTRACTION
# ==========================================
def getZhbbWeight(df_):
    if 'events_norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index)

    weight = df_['events_norm_weight'].copy()
    gen_w = df_.get('events_genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)

    sfs = [
        'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf',
        'events_mu_iso_sf', 'events_puWeight', 'events_topptWeight', 'events_ele_trig_sf', 'events_btag_sf',
    ]
    if not YEAR.startswith("202"):   # UL NanoAOD L1 prefiring weight (Run 2 only)
        sfs.append('events_l1prefire_sf')

    for sf in sfs:
        if sf not in df_.columns:
            print(f"  WARNING: weight column '{sf}' missing, using 1.0")
        weight *= df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)

    return weight


def map_process(raw_proc):
    if 'data' in raw_proc.lower():
        return 'data_obs'
    if 'TTTo2L2Nu' in raw_proc:
        return 'TTBar' if 'tt+B' not in raw_proc else None
    if 'TTbb_2L2Nu' in raw_proc:
        return 'tt_B' if 'tt+B' in raw_proc else None
    if 'TTToSemiLeptonic' in raw_proc:
        return 'TTBar_SemiLep' if 'tt+B' not in raw_proc else None
    if 'TTbb_SemiLeptonic' in raw_proc:
        return 'tt_B_SemiLep' if 'tt+B' in raw_proc else None
    return None


def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR):
    tracked_data = {proc: {} for proc in all_processes}
    files_per_dataset = {}
    skipped_data = set()
    missing_genweight = set()

    # run_pipeline.py layout: <outdir>/nominal/trigeff/output_all.coffea
    all_files = glob.glob(os.path.join(coffea_dir, "output_all.coffea"))
    valid_files = all_files
    print(f"Extracting '{variation}' for {YEAR} from {len(valid_files)} matching files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})

        for raw_proc in filein['columns'].keys():
            mapped_proc = map_process(raw_proc)
            if mapped_proc not in all_processes:
                continue
            is_data = mapped_proc == 'data_obs'

            for dataset in filein['columns'][raw_proc].keys():
                if YEAR not in dataset and (not is_data or FILTER_DATA_BY_YEAR):
                    continue
                if is_data and REF_DATA_PD and REF_DATA_PD not in dataset and REF_DATA_PD not in raw_proc:
                    skipped_data.add(dataset)
                    continue

                genweight = 1.0
                if not is_data:
                    if dataset not in genweight_dict:
                        missing_genweight.add(dataset)
                    genweight = genweight_dict.get(dataset, 1.0)
                    if isinstance(genweight, dict):
                        genweight = genweight.get(dataset, 1.0)

                try:
                    base_dict = filein['columns'][raw_proc][dataset][CATEGORY][variation]
                except KeyError:
                    continue

                tmp_data = {}
                for var in (validation_vars if is_data else vars_to_extract):
                    if var not in base_dict:
                        continue
                    arr = np.array(base_dict[var].value)
                    tmp_data[var] = arr / genweight if (var == 'events_norm_weight' and not is_data) else arr

                if tmp_data:
                    df = pd.DataFrame(tmp_data)
                    df['dataset'] = dataset
                    tracked_data[mapped_proc].setdefault(dataset, []).append(df)
                    files_per_dataset.setdefault((mapped_proc, dataset), set()).add(os.path.basename(file_path))

    # --- Loader sanity report ---
    print("\nDatasets loaded:")
    for (proc, dataset), files in sorted(files_per_dataset.items()):
        print(f"  {proc:14s} {dataset}  ({len(files)} file(s))")
        if proc != 'data_obs' and len(files) > 1:
            print("    WARNING: MC dataset spread over several files. norm_weight is divided by each file's "
                  "'sum_signOf_genweights'; make sure that is the FULL-dataset sum, not a per-file partial sum.")
    if skipped_data:
        print(f"\nSkipped data datasets (not matching REF_DATA_PD='{REF_DATA_PD}'): {sorted(skipped_data)}")
    if missing_genweight:
        print(f"\nWARNING: no sum_signOf_genweights entry for {sorted(missing_genweight)} -> genweight=1 used!")
    if not tracked_data['data_obs']:
        raise RuntimeError(f"No data datasets matched REF_DATA_PD='{REF_DATA_PD}'. "
                           f"Skipped: {sorted(skipped_data)}. Adjust REF_DATA_PD_DICT or set the entry to None.")

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = [df for branches in tracked_data[proc].values() for df in branches]
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            df['dataset'] = df['dataset'].astype('category')
            df['tot_weight'] = getZhbbWeight(df) if proc != 'data_obs' else 1.0
            df['mu_abseta'] = np.abs(df[ETA_RAW])
            data_dict[proc] = df
        else:
            data_dict[proc] = None
    return data_dict


def build_mc_samples(data_dict):
    dilep = [data_dict[p] for p in ['TTBar', 'tt_B'] if data_dict.get(p) is not None]
    semi = [data_dict[p] for p in ['TTBar_SemiLep', 'tt_B_SemiLep'] if data_dict.get(p) is not None]
    if not dilep:
        raise RuntimeError("No dileptonic ttbar MC loaded.")
    data_dict['MC_Combined'] = pd.concat(dilep, ignore_index=True)
    parts = [d.assign(is_semi=False) for d in dilep] + [d.assign(is_semi=True) for d in semi]
    data_dict['MC_WithFakes'] = pd.concat(parts, ignore_index=True)
    if not semi:
        print("WARNING: no semileptonic MC loaded; fake systematic will be zero.")


# ==========================================
# 3. HELPERS
# ==========================================
def trigger_masks(df):
    """NaN-safe ORs. Reference = electron triggers, target = muon triggers.
    A trigger branch missing for some events counts as 'did not fire'."""
    ref = df.reindex(columns=ele_triggers).fillna(False).astype(bool).any(axis=1).values
    tgt = df.reindex(columns=mu_triggers).fillna(False).astype(bool).any(axis=1).values
    acc = (df[PT_COL].between(PT_BINS[0], PT_BINS[-1]) &
           df[ETA_COL].between(ETA_BINS[0], ETA_BINS[-1])).values
    return ref, tgt, acc


def bin_index(df):
    """Flat (pT, eta) bin index per event, -1 if outside the grid. Upper edge goes into the last bin."""
    pt, eta = df[PT_COL].values, df[ETA_COL].values
    ipt = np.searchsorted(PT_BINS, pt, side='right') - 1
    ieta = np.searchsorted(ETA_BINS, eta, side='right') - 1
    ipt[pt == PT_BINS[-1]] = N_PT - 1
    ieta[eta == ETA_BINS[-1]] = N_ETA - 1
    ok = (ipt >= 0) & (ipt < N_PT) & (ieta >= 0) & (ieta < N_ETA)
    return np.where(ok, ipt * N_ETA + ieta, -1)


def hist2d(idx, w, mask):
    m = mask & (idx >= 0)
    return np.bincount(idx[m], weights=w[m], minlength=N_PT * N_ETA).reshape(N_PT, N_ETA)


def safe_ratio(a, b):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(b > 0, a / b, np.nan)


def efficiency_with_cp(h_num, h_den, h_den_w2):
    """Efficiency with Clopper-Pearson interval evaluated at the effective number of entries."""
    with np.errstate(divide='ignore', invalid='ignore'):
        eff = np.where(h_den > 0, h_num / h_den, 0.0)
        n_eff = np.where(h_den_w2 > 0, h_den**2 / h_den_w2, 0.0)
        k_eff = np.clip(eff, 0.0, 1.0) * n_eff
        lo = stats.beta.ppf(CL_LOW, k_eff, n_eff - k_eff + 1)
        hi = stats.beta.ppf(CL_HIGH, k_eff + 1, n_eff - k_eff)
    valid = h_den > 0
    err_up = np.where(valid, np.nan_to_num(hi - eff, nan=0.0), 0.0)
    err_dn = np.where(valid, np.nan_to_num(eff - lo, nan=0.0), 0.0)
    return eff, err_up, err_dn, n_eff


def alpha_from_masks(w, ref, tgt, base):
    S = lambda m: w[m & base].sum()
    return S(ref) * S(tgt) / (S(ref & tgt) * S(base))


# ==========================================
# 4. DIAGNOSTICS
# ==========================================
def trigger_report(data_dict, procs=('data_obs', 'MC_Combined')):
    print("\n=== Trigger branch availability (% of events with branch MISSING, per dataset) ===")
    for proc in procs:
        df = data_dict.get(proc)
        if df is None:
            continue
        miss = pd.DataFrame({t: (df[t].isna() if t in df.columns else pd.Series(True, index=df.index))
                             for t in trigger_vars})
        tab = 100 * miss.groupby(df['dataset'], observed=True).mean()
        if (tab.values > 0).any():
            print(f"[{proc}]")
            print(tab.round(2).to_string())
        else:
            print(f"[{proc}] all trigger branches present in every dataset.")

    print("\n=== Per-path firing rate [%] among reference-triggered events in acceptance ===")
    rows = {}
    for proc in procs:
        df = data_dict.get(proc)
        if df is None:
            continue
        ref, _, acc = trigger_masks(df)
        sel = ref & acc
        w = np.ones(len(df)) if proc == 'data_obs' else df['tot_weight'].values
        rows[proc] = {t: (np.average(df[t].fillna(False).astype(bool).values[sel], weights=w[sel])
                          if t in df.columns else np.nan) for t in mu_triggers}
    print((100 * pd.DataFrame(rows)).round(2).to_string())


def check_eta_symmetry(df):
    """Compare data efficiency for eta>0 vs eta<0 in each |eta| bin."""
    ref, tgt, _ = trigger_masks(df)
    both = ref & tgt
    pt, eta = df[PT_COL].values, df[ETA_RAW].values
    eff, err = {}, {}
    for side, sel in [('pos', eta > 0), ('neg', eta < 0)]:
        h_d, _, _ = np.histogram2d(pt[ref & sel], np.abs(eta[ref & sel]), bins=[PT_BINS, ETA_BINS_SYMCHECK])
        h_n, _, _ = np.histogram2d(pt[both & sel], np.abs(eta[both & sel]), bins=[PT_BINS, ETA_BINS_SYMCHECK])
        with np.errstate(divide='ignore', invalid='ignore'):
            e = np.where(h_d > 0, h_n / h_d, np.nan)
            eff[side] = e
            err[side] = np.sqrt(np.maximum(e * (1 - e), 1.0 / np.maximum(h_d, 1)) / np.maximum(h_d, 1))
    with np.errstate(divide='ignore', invalid='ignore'):
        pull = (eff['pos'] - eff['neg']) / np.sqrt(err['pos']**2 + err['neg']**2)
    print("\nEta symmetry pull (eff[+eta] - eff[-eta]) / sigma  (rows = pT bins, cols = |eta| bins):")
    print(np.round(pull, 2))
    return pull, eff, err


def eta_symmetry_tests(pull):
    ok = np.isfinite(pull)
    chi2, ndf = np.sum(pull[ok]**2), ok.sum()
    p_global = stats.chi2.sf(chi2, ndf)
    print(f"\n[1] Global chi2/ndf = {chi2:.1f}/{ndf}, p = {p_global:.3f}  -> {'OK' if p_global > 0.05 else 'CHECK'}")
    for j in range(pull.shape[1]):
        col = pull[:, j][np.isfinite(pull[:, j])]
        if len(col) == 0:
            continue
        limit = 2 / np.sqrt(len(col))
        print(f"[2] |eta| [{ETA_BINS_SYMCHECK[j]}, {ETA_BINS_SYMCHECK[j+1]}]: mean pull = {col.mean():+.2f} "
              f"(limit ±{limit:.2f}) -> {'OK' if abs(col.mean()) < limit else 'CHECK'}")
    n_extreme = np.sum(np.abs(pull[ok]) > 3)
    print(f"[3] Bins with |pull| > 3: {n_extreme}  -> {'OK' if n_extreme == 0 else 'CHECK'}")


def alpha_vs_pt(df, alpha_incl, n_boot=100, seed=11):
    """Check that alpha is flat in pT, justifying a single inclusive value."""
    ref, tgt, acc = trigger_masks(df)
    w, pt = df['tot_weight'].values, df[PT_COL].values
    rng = np.random.default_rng(seed)
    print(f"\nAlpha vs muon pT (inclusive alpha = {alpha_incl:.4f}):")
    for lo, hi in zip(PT_BINS[:-1], PT_BINS[1:]):
        m = acc & (pt >= lo) & (pt < hi)
        if not m.any():
            continue
        wm, rm, tm, base = w[m], ref[m], tgt[m], np.ones(m.sum(), bool)
        a = alpha_from_masks(wm, rm, tm, base)
        s = np.std([alpha_from_masks(wm * rng.poisson(1.0, len(wm)), rm, tm, base) for _ in range(n_boot)])
        print(f"  pT [{lo:.0f}, {hi:.0f}): alpha = {a:.4f} ± {s:.4f}   (rough pull vs inclusive: {(a - alpha_incl) / s:+.1f})")


def print_uncertainty_breakdown(sf, comps, valid):
    print("\n=== Uncertainty breakdown [% of SF] (rows = pT bins, cols = eta bins) ===")
    for name, m in comps.items():
        with np.errstate(divide='ignore', invalid='ignore'):
            rel = np.where(valid & (sf > 0), 100 * m / sf, np.nan)
        print(f"\n{name}:")
        print(np.round(rel, 2))


# ==========================================
# 5. MEASUREMENT
# ==========================================
def calculate_trigger_sf(data_dict, target_mc='MC_Combined', target_data='data_obs', compute_alpha=True):
    print(f"\n--- Calculating Trigger Efficiencies & Systematics for {target_mc} ---")
    results = {}
    for process in [target_data, target_mc]:
        df = data_dict.get(process)
        if df is None or df.empty:
            raise RuntimeError(f"No events for {process}")

        ref, tgt, acc = trigger_masks(df)
        idx = bin_index(df)
        w = np.ones(len(df)) if process == target_data else df['tot_weight'].values

        h_den = hist2d(idx, w, ref)
        h_den_w2 = hist2d(idx, w**2, ref)
        h_num = hist2d(idx, w, ref & tgt)
        eff, err_up, err_dn, n_eff = efficiency_with_cp(h_num, h_den, h_den_w2)

        results[process] = {'eff': eff, 'stat_err_up': err_up, 'stat_err_dn': err_dn, 'n_eff': n_eff}
        print(f"Calculated efficiency for {process}. Mean eff over filled bins: {eff[h_den > 0].mean():.3f}")

        if process == target_mc and compute_alpha:
            results['alpha'] = alpha_from_masks(w, ref, tgt, acc)
            rng = np.random.default_rng(42)
            boots = [alpha_from_masks(w * rng.poisson(1.0, len(w)), ref, tgt, acc) for _ in range(N_BOOT)]
            results['alpha_err'] = np.std(boots)
            print(f"Inclusive alpha = {results['alpha']:.4f} ± {results['alpha_err']:.4f} (stat)")

    alpha = results.get('alpha', 1.0)

    with np.errstate(divide='ignore', invalid='ignore'):
        eff_data, eff_mc = results[target_data]['eff'], results[target_mc]['eff']
        valid = (results[target_mc]['n_eff'] >= MIN_EFF_STATS) & (results[target_data]['n_eff'] >= MIN_EFF_STATS) & (eff_mc > 0)

        sf_map = np.where(valid, eff_data / eff_mc, 0.0)

        rel_up_data = np.where(eff_data > 0, results[target_data]['stat_err_up'] / eff_data, 0)
        rel_dn_data = np.where(eff_data > 0, results[target_data]['stat_err_dn'] / eff_data, 0)
        rel_up_mc = np.where(eff_mc > 0, results[target_mc]['stat_err_up'] / eff_mc, 0)
        rel_dn_mc = np.where(eff_mc > 0, results[target_mc]['stat_err_dn'] / eff_mc, 0)

        # SF up: data up / MC down ; SF down: data down / MC up
        sf_stat_up = np.where(valid, sf_map * np.sqrt(rel_up_data**2 + rel_dn_mc**2), 0.0)
        sf_stat_dn = np.where(valid, sf_map * np.sqrt(rel_dn_data**2 + rel_up_mc**2), 0.0)

        alpha_unc = np.where(valid, np.abs(1.0 - alpha) * sf_map, 0.0)

    low_stat = ~valid
    if low_stat.any():
        for i, j in zip(*np.where(low_stat)):
            print(f"  -> LOW STATS: pT [{PT_BINS[i]}, {PT_BINS[i+1]}], eta [{ETA_BINS[j]}, {ETA_BINS[j+1]}]: "
                  f"n_eff data = {results[target_data]['n_eff'][i, j]:.1f}, "
                  f"MC = {results[target_mc]['n_eff'][i, j]:.1f}. Consider merging.")
        # Conservative placeholder while iterating on binning. The final binning should have no flagged bins.
        sf_map[low_stat] = 1.0
        sf_stat_up[low_stat] = 1.0
        sf_stat_dn[low_stat] = 1.0
        alpha_unc[low_stat] = 0.0

    return results, sf_map, sf_stat_up, sf_stat_dn, alpha_unc, valid


def eta_asym_syst(eff, err, eff_folded, sf, valid):
    """Half the inverse-variance-weighted, pT-averaged efficiency difference between +eta and -eta,
    per |eta| region, as a relative uncertainty on the SF."""
    d, s2 = eff['pos'] - eff['neg'], err['pos']**2 + err['neg']**2
    w = np.where(np.isfinite(d) & (s2 > 0), 1.0 / s2, 0.0)
    d_col = (w * np.nan_to_num(d)).sum(axis=0) / w.sum(axis=0)     # weighted mean over pT
    d_col_err = 1.0 / np.sqrt(w.sum(axis=0))
    print("\nEta asymmetry per |eta| region (weighted over pT):")
    for j, (dm, de) in enumerate(zip(d_col, d_col_err)):
        print(f"  |eta| [{ETA_BINS[j]}, {ETA_BINS[j+1]}]: eps(+) - eps(-) = {dm:+.4f} ± {de:.4f}")
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(valid & (eff_folded > 0), sf * (np.abs(d_col) / 2)[None, :] / eff_folded, 0.0)


def fake_systematic(data_dict, eff_data, valid, n_boot=N_BOOT, seed=7):
    """SF(dilep MC) - SF(dilep+semilep MC), with the MC-stat noise of that difference
    estimated by a Poisson bootstrap applied coherently to both MC definitions."""
    mc = data_dict['MC_WithFakes']
    if not mc['is_semi'].any():
        return np.zeros_like(eff_data)

    ref, tgt, _ = trigger_masks(mc)
    both, idx = ref & tgt, bin_index(mc)
    dil = ~mc['is_semi'].values
    w = mc['tot_weight'].values

    def delta(wt):
        e_nom = safe_ratio(hist2d(idx, wt, both & dil), hist2d(idx, wt, ref & dil))
        e_alt = safe_ratio(hist2d(idx, wt, both), hist2d(idx, wt, ref))
        return safe_ratio(eff_data, e_nom) - safe_ratio(eff_data, e_alt)

    d = delta(w)
    rng = np.random.default_rng(seed)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        boots = np.array([delta(w * rng.poisson(1.0, len(w))) for _ in range(n_boot)])
        sigma = np.nanstd(boots, axis=0)

    semi_frac = 1 - hist2d(idx, w, ref & dil).sum() / hist2d(idx, w, ref).sum()
    print(f"\nSemileptonic fraction of MC denominator (weighted): {semi_frac:.2%}")
    print("Fake-syst significance, delta / sigma_stat(delta) (rows = pT bins, cols = eta bins):")
    with np.errstate(divide='ignore', invalid='ignore'):
        print(np.round(np.where(valid, d / sigma, np.nan), 1))

    if SUBTRACT_STAT_FROM_FAKE_SYST:
        syst = np.sqrt(np.maximum(d**2 - sigma**2, 0.0))
    else:
        syst = np.abs(d)
    return np.where(valid & np.isfinite(syst), syst, 0.0), semi_frac


# ==========================================
# 6. WORKFLOW
# ==========================================
data_dict = load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR)
build_mc_samples(data_dict)

trigger_report(data_dict)

pull, eff_side, err_side = check_eta_symmetry(data_dict['data_obs'])
eta_symmetry_tests(pull)

results_nom, sf_nominal, sf_stat_up, sf_stat_dn, sf_alpha_err, valid = calculate_trigger_sf(data_dict, target_mc='MC_Combined')
alpha_vs_pt(data_dict['MC_Combined'], results_nom['alpha'])

fake_out = fake_systematic(data_dict, results_nom['data_obs']['eff'], valid)
sf_fake_err, semi_frac = fake_out if isinstance(fake_out, tuple) else (fake_out, 0.0)
sf_eta_err = eta_asym_syst(eff_side, err_side, results_nom['data_obs']['eff'], sf_nominal, valid)
sf_syst = np.sqrt(sf_alpha_err**2 + sf_fake_err**2 + sf_eta_err**2)

sf_total_err_up = np.sqrt(sf_stat_up**2 + sf_syst**2)
sf_total_err_dn = np.sqrt(sf_stat_dn**2 + sf_syst**2)

print_uncertainty_breakdown(sf_nominal, {
    'stat up': sf_stat_up, 'stat down': sf_stat_dn,
    'alpha (correlation)': sf_alpha_err, 'fake (semilep)': sf_fake_err,
    'eta asymmetry': sf_eta_err,
    'total up': sf_total_err_up, 'total down': sf_total_err_dn,
}, valid)

print("\nData n_eff per bin (rows = pT bins, cols = eta bins):")
print(np.round(results_nom['data_obs']['n_eff'], 0))
print("MC n_eff per bin (rows = pT bins, cols = eta bins):")
print(np.round(results_nom['MC_Combined']['n_eff'], 0))

# --- Numbers for the AN ---
ok = np.isfinite(pull)
chi2, ndf = np.sum(pull[ok]**2), ok.sum()
col_mean, n_col = np.nanmean(pull, axis=0), np.isfinite(pull).sum(axis=0)
rel = lambda m: 100 * np.where(valid, m / sf_nominal, np.nan)
print(f"\n=== AN numbers (Muon, {YEAR}) ===")
print(f"chi2/ndf = {chi2:.1f}/{ndf}, p = {stats.chi2.sf(chi2, ndf):.2f}, max|d| = {np.abs(pull[ok]).max():.2f}")
print(f"mean-pull limit = ±{2 / np.sqrt(N_PT):.2f} ({N_PT} pT bins)")
print("mean pull per |eta| region: " + ", ".join(f"{m:+.2f} ({m*np.sqrt(n):+.1f} sigma)" for m, n in zip(col_mean, n_col)))
print(f"alpha = {results_nom['alpha']:.4f} ± {results_nom['alpha_err']:.4f}")
print(f"semileptonic fraction = {100 * semi_frac:.2f}%")
for name, m in [('stat up', sf_stat_up), ('stat down', sf_stat_dn), ('fake', sf_fake_err), ('eta asym', sf_eta_err)]:
    print(f"{name:10s}: {np.nanmin(rel(m)):.2f}% – {np.nanmax(rel(m)):.2f}%")


# ==========================================
# 7. PLOTTING & EXPORT
# ==========================================
def plot_annotated_heatmap(val, err_up, err_dn, title, filename, is_sf=False, low_stat=None, data_label=True):
    fig, ax = plt.subplots(figsize=(14, 8))
    X, Y = np.meshgrid(PT_BINS, ETA_BINS)

    shown = val.astype(float).copy()
    if low_stat is not None:
        shown[low_stat] = np.nan

    vmin, vmax, cmap = 0.0, 1.0, 'viridis'
    label = "SF (Data/MC)" if is_sf else "Efficiency"

    im = ax.pcolormesh(X, Y, np.ma.masked_invalid(shown).T, cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(label, fontsize=18)

    ax.set_xscale('log')
    ax.set_xticks(PT_BINS)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([f"{pt:.0f}" for pt in PT_BINS])
    ax.minorticks_off()
    ax.set_xlabel(r"Muon $p_T$ [GeV]", fontsize=18)
    ax.set_yticks(ETA_BINS)
    ax.set_yticklabels([f"{eta:.1f}" for eta in ETA_BINS])
    ax.set_ylabel(ETA_LABEL, fontsize=18)

    for i in range(N_PT):
        for j in range(N_ETA):
            x_c = np.exp((np.log(PT_BINS[i]) + np.log(PT_BINS[i + 1])) / 2.0)
            y_c = (ETA_BINS[j] + ETA_BINS[j + 1]) / 2.0
            if low_stat is not None and low_stat[i, j]:
                ax.text(x_c, y_c, "low\nstats", ha='center', va='center', color='gray', fontsize=10)
                continue
            if val[i, j] <= 0:
                continue
            color = 'white' if (not is_sf and (val[i, j] - vmin) / (vmax - vmin) < 0.5) else 'black'
            text = f"{val[i, j]:.3f}\n+{err_up[i, j]:.3f}\n-{err_dn[i, j]:.3f}"
            ax.text(x_c, y_c, text, ha='center', va='center', color=color, fontsize=10, fontweight='bold')

    fig.suptitle(title, fontsize=22, fontweight='bold', y=0.94)
    hep.cms.label("Preliminary", data=data_label, lumi=LUMI, ax=ax)
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()


low_stat = ~valid
print("\nGenerating Heatmaps...")
plot_annotated_heatmap(results_nom['data_obs']['eff'], results_nom['data_obs']['stat_err_up'],
                       results_nom['data_obs']['stat_err_dn'], f"Data Muon Trigger Efficiency ({YEAR})",
                       f"pdf/trigger/Data_Mu_Eff_{YEAR}.pdf", low_stat=low_stat)
plot_annotated_heatmap(results_nom['MC_Combined']['eff'], results_nom['MC_Combined']['stat_err_up'],
                       results_nom['MC_Combined']['stat_err_dn'], f"MC Muon Trigger Efficiency ({YEAR})",
                       f"pdf/trigger/MC_Mu_Eff_{YEAR}.pdf", low_stat=low_stat, data_label=False)
plot_annotated_heatmap(sf_nominal, sf_total_err_up, sf_total_err_dn, f"Muon Trigger Scale Factors ({YEAR})",
                       f"pdf/trigger/Mu_SF_{YEAR}.pdf", is_sf=True, low_stat=low_stat)


def export_to_correctionlib():
    input_pt = schema.Variable(name="pt", type="real", description="Muon pT")
    input_eta = schema.Variable(name="eta", type="real", description="Muon pseudorapidity (signed)")
    input_syst = schema.Variable(
        name="systematic", type="string",
        description="'nominal', 'up'/'down' (total), 'stat_up'/'stat_down', 'syst_up'/'syst_down' "
                    "(alpha + semilep fake + eta asymmetry, symmetric). Stat and syst can be decorrelated across years."
    )

    variations = {
        "nominal": sf_nominal,
        "up": sf_nominal + sf_total_err_up,
        "down": np.clip(sf_nominal - sf_total_err_dn, 0.0, None),
        "stat_up": sf_nominal + sf_stat_up,
        "stat_down": np.clip(sf_nominal - sf_stat_dn, 0.0, None),
        "syst_up": sf_nominal + sf_syst,
        "syst_down": np.clip(sf_nominal - sf_syst, 0.0, None),
    }

    def build_multibin(val_matrix):
        # If measured in |eta|, mirror back to signed eta so consumers pass signed eta
        if USE_ABS_ETA:
            edges_eta = np.concatenate([-ETA_BINS[:0:-1], ETA_BINS])
            content = np.concatenate([val_matrix[:, ::-1], val_matrix], axis=1)
        else:
            edges_eta, content = ETA_BINS, val_matrix
        return schema.MultiBinning(nodetype="multibinning", inputs=["pt", "eta"],
                                   edges=[PT_BINS.tolist(), edges_eta.tolist()],
                                   content=content.flatten().tolist(), flow="clamp")

    cat = schema.Category(nodetype="category", input="systematic", content=[
        schema.CategoryItem(key=k, value=build_multibin(v)) for k, v in variations.items()
    ])

    desc = (f"Muon trigger SF {YEAR}, electron-reference (orthogonal) method in ttbar e-mu. "
            f"Inclusive alpha = {results_nom['alpha']:.4f} ± {results_nom['alpha_err']:.4f}.")
    cset = schema.CorrectionSet(schema_version=2, description=desc, corrections=[
        schema.Correction(name="MuTrigSF", description=desc, version=1,
                          inputs=[input_pt, input_eta, input_syst],
                          output=schema.Variable(name="weight", type="real"), data=cat)
    ])

    fname = f"muon_trig_sf_{YEAR}.json.gz"
    with gzip.open(fname, "wt") as fout:
        fout.write(cset.json(exclude_unset=True))
    print(f"\nSaved correctionlib to: {fname}")


export_to_correctionlib()